In [172]:
import requests
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import datetime as dt
import itertools
import json
import pymysql
import geopandas as gpd
import yaml
import os

In [173]:
def parse_onboarding_time(t):
    try:
        t_str = str(int(t)).zfill(12)
        return datetime.strptime(t_str, "%Y%m%d%H%M")
    except:
        return np.nan

In [174]:
current_mode = "dynamic"

In [175]:
con = pymysql.connect(
        user = "root",
        passwd = "3644",
        host = "143.248.121.90",
        port = 3306,
        db = "hdl",
        charset = "utf8mb4",
        use_unicode = True
)
mycursor = con.cursor()
query = """
    select * from reservation_request;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
request_df = pd.DataFrame(data, columns=["requestID", "passengerID", "messageTime", "pickupStationID", "dropoffStationID", "serviceType", "reserveType", "dispatchID", "responseStatus", "confirmCheck", "passengerCount", "wheelchairCount", "failInfoList", "pickupTimeRequest"])


In [176]:
con = pymysql.connect(
        user = "root",
        passwd = "3644",
        host = "143.248.121.90",
        port = 3306,
        db = "hdl",
        charset = "utf8mb4",
        use_unicode = True
)
mycursor = con.cursor()
query = """
    select * from dispatch;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
dispatch_df = pd.DataFrame(data, columns=["dispatchID", "messageTime", "passengerID", "requestID", "routeIDs", "pickupStationName", "dropoffStationName", "reserveType", "onboardingTime", "dropoffTime", "linkIDs", "pickupStationID", "dropoffStationID", "tripID", "operationID", "vehicleID"])

In [177]:
con = pymysql.connect(
        user = "root",
        passwd = "3644",
        host = "143.248.121.90",
        port = 3306,
        db = "hdl",
        charset = "utf8mb4",
        use_unicode = True
)
mycursor = con.cursor()
query = """
    select * from operation;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
operation_df = pd.DataFrame(data, columns=["operationID", "vehicleID", "StationIDs", "routeIDs", "startTime", "endTime", "VehicleType", "operationServiceType"])


In [178]:
con = pymysql.connect(
        user = "root",
        passwd = "3644",
        host = "143.248.121.90",
        port = 3306,
        db = "hdl",
        charset = "utf8mb4",
        use_unicode = True
)
mycursor = con.cursor()
query = """
    select * from route;
"""
mycursor.execute(query)
data = mycursor.fetchall()
con.close()
route_df = pd.DataFrame(data, columns=["routeID", "routeSeq", "operationID", "vehicleID", "routeInfo", "linkIDs", "NodeIDs", "originStationID", "originDeptTime", "destinationID", "onboardingNum", "dispatchIDs", "lon", "lat", "originBoardingPxIDs", "originGetoffPxIDs", "destBoardingPxIDs", "destGetoffPxIDs", "destArrivalTime", "routeCode"])


In [179]:
request_df['messageTime_Request'] = request_df['messageTime']
request_df['messageTime_Request'] = pd.to_datetime(request_df['messageTime_Request'], unit='ms', utc=True)

In [180]:
dispatch_df['onboarding_datetime'] = dispatch_df['onboardingTime'].apply(parse_onboarding_time)
dispatch_df['dropoff_datetime'] = dispatch_df['dropoffTime'].apply(parse_onboarding_time)

operation_df['startTime_datetime'] = operation_df['startTime'].apply(parse_onboarding_time)
operation_df['endTime_datetime'] = operation_df['endTime'].apply(parse_onboarding_time)

route_df['originDeptTime_datetime'] = route_df['originDeptTime'].apply(parse_onboarding_time)
route_df['destArrivalTime_datetime'] = route_df['destArrivalTime'].apply(parse_onboarding_time)

reservetype_dict = {
    '사전 예약': 1,
    '실시간 예약' : 2,
}

In [181]:
current_time = datetime.now()
days_interval = 7
sevice_Type=[1, 2]

In [182]:
temp_operation_df = operation_df[(operation_df['endTime_datetime'] >= current_time - dt.timedelta(days=days_interval*2)) & (operation_df['endTime_datetime'] < current_time)].sort_values("endTime_datetime").reset_index(drop=True)
temp_operation_df['Day'] = [(temp_operation_df['endTime_datetime'][i] - current_time).days for i in range(len(temp_operation_df))]
temp_operation_df['Hour'] = [temp_operation_df['endTime_datetime'][i].hour for i in range(len(temp_operation_df))]
temp_operation_df['Operation_vehicle'] = [str(temp_operation_df['operationID'][i]) + '_' + str(temp_operation_df['vehicleID'][i]) for i in range(len(temp_operation_df))]
temp_operation_df = temp_operation_df[['Operation_vehicle', 'VehicleType', 'Day', 'Hour']]
dispatch_df['Operation_vehicle'] = [str(dispatch_df['operationID'][i]) + '_' + str(dispatch_df['vehicleID'][i]) for i in range(len(dispatch_df))]
temp_operations = temp_operation_df.Operation_vehicle.tolist()
temp_dispatch_df = dispatch_df[dispatch_df['Operation_vehicle'].isin(temp_operations)].reset_index(drop=True)
temp_merged = pd.merge(left = temp_dispatch_df , right = temp_operation_df, how = "inner", on = "Operation_vehicle")

In [183]:
temp_dispatchIDs = temp_merged.dispatchID.tolist()
temp_request_df = request_df[request_df['dispatchID'].isin(temp_dispatchIDs)].reset_index(drop=True)
temp_request_df['messageTime_Request'] = temp_request_df['messageTime']
temp_request_df['messageTime_Request'] = pd.to_datetime(temp_request_df['messageTime_Request'], unit='ms', utc=True)
temp_request_df['messageTime_Request'] = temp_request_df['messageTime_Request'].dt.tz_convert('Asia/Seoul')

In [184]:
temp_request_df = temp_request_df[['dispatchID', 'messageTime_Request', 'pickupTimeRequest', 'serviceType']]
final_merged = pd.merge(left = temp_merged , right = temp_request_df, how = "inner", on = "dispatchID").sort_values('Day').reset_index(drop=True)
final_merged = final_merged[final_merged['serviceType'].isin(sevice_Type)].sort_values('Day').reset_index(drop=True)

In [185]:
final_merged[['requestID', 'onboarding_datetime','pickupTimeRequest', 'reserveType', 'messageTime_Request']].head(20)

,requestID,onboarding_datetime,pickupTimeRequest,reserveType,messageTime_Request
0,req-0044,2026-01-16 15:31:00,1530.0,1,2026-01-15 12:28:09+09:00
1,req-0011,2026-01-16 12:04:00,1200.0,1,2026-01-15 09:39:04+09:00
2,req-0184,2026-01-16 21:11:00,2100.0,1,2026-01-16 12:13:05+09:00
3,req-0146,2026-01-16 09:33:00,1700.0,1,2026-01-16 09:24:07+09:00
4,req-0016,2026-01-16 17:31:00,1730.0,1,2026-01-15 10:10:01+09:00
5,req-0073,2026-01-16 19:04:00,1900.0,1,2026-01-15 14:59:05+09:00
6,req-0036,2026-01-16 12:33:00,1230.0,1,2026-01-15 11:54:08+09:00
7,req-0178,2026-01-16 11:52:00,NaN,2,2026-01-16 11:29:06+09:00
8,req-0021,2026-01-16 16:33:00,1630.0,1,2026-01-15 10:48:08+09:00
9,req-0141,2026-01-16 09:17:00,NaN,2,2026-01-16 09:06:08+09:00


In [186]:
temp_operation_df = operation_df[(operation_df['endTime_datetime'] >= current_time - dt.timedelta(days=days_interval*2)) & (operation_df['endTime_datetime'] < current_time)].sort_values("endTime_datetime").reset_index(drop=True)
temp_operation_df['Day'] = [(temp_operation_df['endTime_datetime'][i] - current_time).days for i in range(len(temp_operation_df))]
temp_operation_df['Hour'] = [temp_operation_df['endTime_datetime'][i].hour for i in range(len(temp_operation_df))]
temp_operation_df['Operation_vehicle'] = [str(temp_operation_df['operationID'][i]) + '_' + str(temp_operation_df['vehicleID'][i]) for i in range(len(temp_operation_df))]
temp_operation_df = temp_operation_df[['Operation_vehicle', 'VehicleType', 'Day', 'Hour']]
dispatch_df['Operation_vehicle'] = [str(dispatch_df['operationID'][i]) + '_' + str(dispatch_df['vehicleID'][i]) for i in range(len(dispatch_df))]
temp_operations = temp_operation_df.Operation_vehicle.tolist()
temp_dispatch_df = dispatch_df[dispatch_df['Operation_vehicle'].isin(temp_operations)].reset_index(drop=True)
temp_merged = pd.merge(left = temp_dispatch_df , right = temp_operation_df, how = "inner", on = "Operation_vehicle")

temp_dispatchIDs = temp_merged.dispatchID.tolist()
temp_request_df = request_df[request_df['dispatchID'].isin(temp_dispatchIDs)].reset_index(drop=True)
temp_request_df['messageTime_Request'] = temp_request_df['messageTime']
temp_request_df['messageTime_Request'] = pd.to_datetime(temp_request_df['messageTime_Request'], unit='ms', utc=True)
temp_request_df['messageTime_Request'] = temp_request_df['messageTime_Request'].dt.tz_convert('Asia/Seoul')

temp_request_df = temp_request_df[['dispatchID', 'messageTime_Request', 'pickupTimeRequest', 'serviceType']]
final_merged = pd.merge(left = temp_merged , right = temp_request_df, how = "inner", on = "dispatchID").sort_values('Day').reset_index(drop=True)
final_merged = final_merged[final_merged['serviceType'].isin(sevice_Type)].sort_values('Day').reset_index(drop=True)
final_merged['messageTime'] = pd.to_datetime(final_merged['messageTime'], unit='ms', utc=True)
final_merged['messageTime'] = final_merged['messageTime'].dt.tz_convert('Asia/Seoul')
response_gaps = [(final_merged['messageTime'][i] - final_merged['messageTime_Request'][i]).total_seconds() for i in range(len(final_merged))]
response_gaps = [x if x>0 else x+9*60*60 for x in response_gaps]
final_merged['Response_time'] = response_gaps

In [187]:
final_merged['messageTime_Request'] = (
    pd.to_datetime(final_merged['messageTime_Request'], utc=True)
    + pd.Timedelta(hours=9)
).dt.tz_localize(None)

In [188]:
pickup_time_list = []
for i in range(len(final_merged)):

    try:
        temp_pickup_time = str(int(final_merged['pickupTimeRequest'][i]))
        temp_onboarding_datetime = final_merged['onboarding_datetime'][i]
        time_str = str(temp_pickup_time)
        converted_time = dt.datetime.strptime("{}-{}-{} {}:{}".format(str(temp_onboarding_datetime.year), str(temp_onboarding_datetime.month), str(temp_onboarding_datetime.day), int(time_str[0:2]), int(time_str[2:4])), "%Y-%m-%d %H:%M")
        pickup_time_list.append(converted_time)
    except:
        pickup_time_list.append(np.nan)

final_merged['pickup_request_datetime'] = pickup_time_list

In [189]:
final_merged['Waiting_Time'] = [(final_merged['onboarding_datetime'][i] - final_merged['messageTime_Request'][i]).total_seconds()/60 if final_merged['reserveType'][i] == 2 else (final_merged['onboarding_datetime'][i].tz_localize("Asia/Seoul") - final_merged['pickup_request_datetime'][i].tz_localize("Asia/Seoul")).total_seconds()/60 for i in range(len(final_merged))]
final_merged = final_merged[(final_merged['Waiting_Time']>=0)&(final_merged['Waiting_Time']<60)].reset_index(drop=True)

In [190]:
final_merged[['onboarding_datetime', 'pickup_request_datetime', 'Waiting_Time']]

,onboarding_datetime,pickup_request_datetime,Waiting_Time
0,2026-01-16 15:31:00,2026-01-16 15:30:00,1.00
1,2026-01-16 12:04:00,2026-01-16 12:00:00,4.00
2,2026-01-16 21:11:00,2026-01-16 21:00:00,11.00
3,2026-01-16 17:31:00,2026-01-16 17:30:00,1.00
4,2026-01-16 19:04:00,2026-01-16 19:00:00,4.00
...,...,...,...
315,2026-01-29 20:30:00,2026-01-29 20:30:00,0.00
316,2026-01-29 18:31:00,2026-01-29 18:30:00,1.00
317,2026-01-29 20:30:00,2026-01-29 20:30:00,0.00
318,2026-01-29 20:00:00,2026-01-29 20:00:00,0.00


In [191]:
final_merged['StationID'] = final_merged['pickupStationID']
temp_gdf = gdf[['StationID', 'StationLat', 'StationLon']]
FFinal_merged = pd.merge(left = final_merged, right = temp_gdf, how = "inner", on = "StationID")
FFinal_merged['Use_Time'] = [(FFinal_merged['dropoff_datetime'][i] - FFinal_merged['onboarding_datetime'][i]).total_seconds()/60 for i in range(len(FFinal_merged))]
FFinal_merged = FFinal_merged[["Day", "reserveType", "VehicleType", "StationID", "StationLat", "StationLon", "Response_time", "Waiting_Time", "Use_Time"]]

past_df = FFinal_merged[final_merged['Day'] < -days_interval].reset_index(drop=True)
last_df = FFinal_merged[FFinal_merged['Day'] >= -days_interval].reset_index(drop=True)

if reserveType is not None:
    last_df_type = last_df[last_df['reserveType'] == reservetype_dict[reserveType]]
    last_df_type = last_df_type.sort_values('StationLon', ascending=False).reset_index(drop=True)
    last_df_type['Waiting_Time'] = (last_df_type['Waiting_Time'])

    locations = [
        {"lat": row.StationLat, "lng": row.StationLon, "weight": np.round(row['Waiting_Time'], 1), "station": row['StationID']}
        for _, row in last_df_type.iterrows()
    ]
else:
    last_df_type = None
    locations = None

mean_last_response_time = np.nanmean(last_df['Response_time'])
mean_past_response_time = np.nanmean(past_df['Response_time'])
mean_last_waiting_time = np.nanmean(last_df['Waiting_Time'])
mean_past_waiting_time = np.nanmean(past_df['Waiting_Time'])
mean_last_use_time = np.nanmean(last_df['Use_Time'])
mean_past_use_time = np.nanmean(past_df['Use_Time'])

NameError: name 'gdf' is not defined

In [192]:
sejong_gdf = gpd.read_file("../data/ODD/convenience/Station_Convenience.shp")
daejeon_gdf = gpd.read_file("../data/ODD/vulnerable/Station_Vulnerable.shp")

c:\Users\Justin\anaconda3\envs\MOVE\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: One or several characters couldn't be converted correctly from EUC-KR to UTF-8.  This warning will not be emitted anymore
  return ogr_read(
c:\Users\Justin\anaconda3\envs\MOVE\Lib\site-packages\pyogrio\raw.py:198: RuntimeWarning: One or several characters couldn't be converted correctly from EUC-KR to UTF-8.  This warning will not be emitted anymore
  return ogr_read(


In [197]:
station_df = pd.concat([sejong_gdf, daejeon_gdf]).reset_index(drop=True)
station_df['pickupStationID'] = station_df['StationID']

In [198]:
station_df

,StationID,StationLat,StationLon,StationX,StationY,StaName,SwalkWidth,existence,CurbHeight,CurbType,...,StationDis,RStationLt,RStationLn,RStationX,RStationY,StationEli,Wheelchair,Remark,geometry,pickupStationID
0,S233000020,37.219991,126.807104,305435,4121530,None,1.6,0,0.00,NaN,...,607.064,NaN,NaN,NaN,NaN,False,False,None,POINT (126.8071 37.21999),S233000020
1,S233000030,37.219630,126.807428,305463,4121489,None,2.8,0,0.00,NaN,...,25.242,37.219624,126.807407,305461.0,4121489.0,True,True,None,POINT (126.80743 37.21963),S233000030
2,S233000040,37.218846,126.808310,305540,4121400,None,2.1,0,0.00,NaN,...,143.673,37.218841,126.808299,305539.0,4121400.0,True,True,None,POINT (126.80831 37.21885),S233000040
3,S233000050,37.218375,126.808995,305599,4121347,None,2.9,0,0.00,NaN,...,225.437,37.218359,126.808985,305598.0,4121345.0,True,True,None,POINT (126.809 37.21837),S233000050
4,S233000060,37.218213,126.810180,305704,4121326,None,1.9,0,0.00,NaN,...,339.173,37.218200,126.810189,305705.0,4121325.0,True,True,None,POINT (126.81018 37.21821),S233000060
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1710,S233009210,37.253535,126.816466,306352,4125233,?1由?,0.0,0,0.00,NaN,...,9.908,37.253510,126.816498,306353.0,4125229.0,True,False,None,POINT (126.81647 37.25353),S233009210
1711,S233009230,37.232227,126.804268,305215,4122893,?ㅺ媛?,0.0,0,0.00,NaN,...,21.737,37.232223,126.804231,305212.0,4122893.0,True,True,None,POINT (126.80427 37.23223),S233009230
1712,S233009220,37.232236,126.804300,305218,4122894,?ㅺ媛?,0.0,0,0.00,NaN,...,77.309,37.232243,126.804349,305222.0,4122895.0,True,False,None,POINT (126.8043 37.23224),S233009220
1713,S233009240,37.199730,126.823454,306835,4119248,????1李?,5.0,1,0.15,3.0,...,17.409,37.199735,126.823431,306833.0,4119249.0,True,False,None,POINT (126.82345 37.19973),S233009240


In [ ]:
temp_dispatch_df = dispatch_df[(dispatch_df['onboarding_datetime'] >= current_time - dt.timedelta(days=days_interval*2)) & (dispatch_df['onboarding_datetime'] < current_time)].sort_values("onboarding_datetime").reset_index(drop=True)
temp_dispatch_df = temp_dispatch_df[['onboarding_datetime', 'reserveType', 'dispatchID', 'pickupStationID', 'vehicleID']]
temp_dispatch_df['Day'] = [(temp_dispatch_df['onboarding_datetime'][i] - current_time).days for i in range(len(temp_dispatch_df))]
temp_dispatch_df['Hour'] = [temp_dispatch_df['onboarding_datetime'][i].hour for i in range(len(temp_dispatch_df))]

temp_station_df = station_df[['pickupStationID', 'StationLat', 'StationLon']]
temp_merged_df = pd.merge(left = temp_dispatch_df , right = temp_station_df, how = "inner", on = "pickupStationID")
temp_merged_df['vehicleType'] = [vehicle_dict[vehid] for vehid in temp_merged_df['vehicleID'].tolist()]

temp_request_df = request_df[['dispatchID', 'passengerCount', 'wheelchairCount', 'serviceType']]
final_merged_df = pd.merge(left = temp_merged_df , right = temp_request_df, how = "inner", on = "dispatchID")

past_df = final_merged_df[final_merged_df['Day'] < -days_interval].reset_index(drop=True)
last_df = final_merged_df[final_merged_df['Day'] >= -days_interval].reset_index(drop=True)

last_df['geometry'] = last_df.apply(lambda row: Point(row['StationLon'], row['StationLat']), axis=1)
last_gdf = gpd.GeoDataFrame(last_df, geometry='geometry', crs="EPSG:4326")

wheelchair_df = last_gdf[last_gdf['wheelchairCount'] == 1]
joined_disabled = gpd.sjoin(wheelchair_df, pop_df, how="left", predicate="within")

older_df = last_gdf[last_gdf['wheelchairCount'] == 0]
joined_older = gpd.sjoin(older_df, pop_df, how="left", predicate="within")

disabled_counts = joined_disabled.groupby('gid')['passengerCount'].sum().reset_index()
disabled_counts.rename(columns={'passengerCount': 'pickup_disabledCount'}, inplace=True)

older_counts = joined_older.groupby('gid')['passengerCount'].sum().reset_index()
older_counts.rename(columns={'passengerCount': 'pickup_olderadultsCount'}, inplace=True)

final_pop_df = pop_df.merge(disabled_counts, on='gid', how='left')
final_pop_df = final_pop_df.merge(older_counts, on='gid', how='left')

final_pop_df['pickup_disabledCount'] = final_pop_df['pickup_disabledCount'].fillna(0).astype(int)
final_pop_df['pickup_olderadultsCount'] = final_pop_df['pickup_olderadultsCount'].fillna(0).astype(int)
final_pop_df['pickup_totalCount'] = final_pop_df['pickup_disabledCount'] + final_pop_df['pickup_olderadultsCount']

final_pop_df['disabled_percent'] = (final_pop_df['pickup_disabledCount'] / final_pop_df['disabled']) * 0.9
final_pop_df['olderadults_percent'] = (final_pop_df['pickup_olderadultsCount'] / final_pop_df['older_adul']) * 0.9
final_pop_df['total_percent'] = final_pop_df['disabled_percent'] + final_pop_df['olderadults_percent'] * 0.9

final_pop_df = final_pop_df[['disabled_percent', 'olderadults_percent', 'total_percent', 'geometry']]

AttributeError: 'GeoDataFrame' object has no attribute 'display'